# Process PB1 DMS data from Li et al.

In [1]:
import os
import pandas as pd
from collections import defaultdict
from Bio import SeqIO
import subprocess

In [2]:
import yaml

_config_dir = os.path.dirname(os.path.abspath("../config.yaml"))
with open("../config.yaml") as _f:
    _config = yaml.safe_load(_f)
data_dir = os.path.normpath(os.path.join(_config_dir, _config["data_dir"]))

Align the unmutated DMS sequence and the sequence used as reference for tree building. See if there are any gaps.

In [3]:
# Read in the sequence of the reference PB1 used to build the tree
ref_fasta = os.path.join(data_dir, 'PB1/all/curated_reference.fasta')
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
ref_aa_seq = ref_aa_seq.replace('*', '')
print('reference sequence length:', len(ref_aa_seq))

# Read in the DMS data
# The file has 2 header rows before the actual data
# wildtype and substitution columns use 3-letter amino acid codes
three_to_one = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V',
}

raw_dms = pd.read_csv('../data/dms_data/Li_PB1/jvi.01329-23-s0008.csv', skiprows=2)

# Get the sequence of the unmutated protein from the DMS experiment.
# Build from raw data (before dms_effect filtering) so all sites are represented.
# Drop the stop codon site (wildtype='***') before mapping to 1-letter codes.
dms_seq_df = (
    raw_dms
    .loc[raw_dms['wildtype'] != '***']
    .assign(wt_aa=lambda df: df['wildtype'].map(three_to_one))
    .rename(columns={'site': 'dms_site'})
    .sort_values('dms_site')
    .drop_duplicates('dms_site')
)

# Check that there are no gaps in the site numbering
n_sites = len(dms_seq_df)
expected_n_sites = dms_seq_df['dms_site'].max() - dms_seq_df['dms_site'].min() + 1
assert n_sites == expected_n_sites, f'Site gaps detected: {n_sites} unique sites but expected {expected_n_sites}'

# Build the filtered mutation data (exclude stop codons and missing fitness values)
pb1_dms_data = (
    raw_dms
    .assign(
        wt_aa=lambda df: df['wildtype'].map(three_to_one),
        mut_aa=lambda df: df['substitution'].map(three_to_one),
    )
    .rename(columns={'site': 'dms_site', 'fitness': 'dms_effect'})
    .dropna(subset=['wt_aa', 'mut_aa', 'dms_effect'])
    [['dms_site', 'wt_aa', 'mut_aa', 'dms_effect']]
)

aa_seq = ''.join(list(dms_seq_df['wt_aa']))
print('DMS sequence length:', len(aa_seq))
print('DMS site range:', dms_seq_df['dms_site'].min(), '-', dms_seq_df['dms_site'].max())

# Save sequences to a FASTA file for alignment
output_dir = '../results/dms_data/Li_PB1/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n')
        f.write(ref_aa_seq + '\n')
        f.write('>dms\n')
        f.write(aa_seq + '\n')

# Align the sequences using MUSCLE
aligned_fasta = os.path.join(output_dir, 'aligned.fasta')
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

reference sequence length: 757
DMS sequence length: 757
DMS site range: 1 - 757


In [4]:
# Read in the aligned sequences
seqs_dict = {}
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

# Compute percent identity
n_sites_to_compare = 0
n_identical = 0
for (ref_aa, dms_aa) in zip(aligned_ref_seq, aligned_dms_seq):
    if ref_aa != '-' and dms_aa != '-':
        n_sites_to_compare += 1
        if ref_aa == dms_aa:
            n_identical += 1

print(n_sites_to_compare, n_identical, n_identical / n_sites_to_compare)

# Determine numbering scheme
dms_start_index = int(dms_seq_df['dms_site'].min())
numbering_dict = defaultdict(list)
assert len(aligned_ref_seq) == len(aligned_dms_seq)
for (seq_id, seq, start_index) in [
    ('tree_reference_site', aligned_ref_seq, 1),
    ('dms_site', aligned_dms_seq, dms_start_index),
]:
    seq_index = start_index
    for (alignment_index, aa) in enumerate(seq, 1):
        if aa != '-':
            numbering_dict['alignment_index'].append(alignment_index)
            numbering_dict['seq_id'].append(seq_id)
            numbering_dict['seq_index'].append(seq_index)
            numbering_dict['seq_aa'].append(aa)
            seq_index += 1

alignment_numbering_df = (
    pd.DataFrame(numbering_dict)
    .pivot(index='alignment_index', columns='seq_id', values='seq_index')
    .reset_index()
    .rename_axis(None, axis=1)
    .dropna()
    [['dms_site', 'tree_reference_site']]
)
alignment_numbering_df['dms_site'] = alignment_numbering_df['dms_site'].astype(int)
alignment_numbering_df['tree_reference_site'] = alignment_numbering_df['tree_reference_site'].astype(int)
alignment_numbering_df.head()

757 735 0.9709379128137384


,dms_site,tree_reference_site
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


In [5]:
# Merge DMS data with numbering scheme and save
pb1_dms_data_processed = (
    pb1_dms_data
    .merge(alignment_numbering_df, on='dms_site', validate='many_to_one')
)

pb1_dms_data_processed = pb1_dms_data_processed[['dms_site', 'wt_aa', 'mut_aa', 'tree_reference_site', 'dms_effect']]
pb1_dms_data_processed.to_csv(os.path.join(output_dir, 'processed_dms_data.csv'), index=False)
print('Number of mutations with processed data:', len(pb1_dms_data_processed))
pb1_dms_data_processed.head()

Number of mutations with processed data: 12760


,dms_site,wt_aa,mut_aa,tree_reference_site,dms_effect
0,1,M,A,1,-1.958803
1,1,M,R,1,-1.176287
2,1,M,N,1,-1.828807
3,1,M,D,1,-1.772591
4,1,M,C,1,-1.371866


In [6]:
pb1_dms_data_processed.tail()

,dms_site,wt_aa,mut_aa,tree_reference_site,dms_effect
12755,757,K,S,757,-0.359050
12756,757,K,T,757,-0.836881
12757,757,K,W,757,-0.893345
12758,757,K,Y,757,-0.831823
12759,757,K,V,757,0.099726
